In [1]:
# Cell 1: Import library yang diperlukan
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import datetime
import warnings
warnings.filterwarnings('ignore')

2026-05-13 13:04:17.689708: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778677457.713157     202 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778677457.721028     202 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778677457.739648     202 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778677457.739675     202 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778677457.739678     202 computation_placer.cc:177] computation placer alr

In [2]:
# Cell 2: Baca file CSV dan lihat distribusi gaya belajar
tf.random.set_seed(42)
np.random.seed(42)

lokasi_file = '/kaggle/input/datasets/nelsonferdinandw/capstone/Data_Final.csv'
data = pd.read_csv(lokasi_file)
print(f"Total data: {data.shape[0]} sampel")
print(data['Dominant_Style'].value_counts())

Total data: 1130 sampel
Dominant_Style
Visual         345
Auditory       293
ReadWrite      252
Kinesthetic    240
Name: count, dtype: int64


In [3]:
# Cell 3: Pisahkan fitur (20 kolom V1..K5) dan target, lalu encoding label
fitur_kolom = [f'V{i}' for i in range(1,6)] + [f'A{i}' for i in range(1,6)] + [f'R{i}' for i in range(1,6)] + [f'K{i}' for i in range(1,6)]
X = data[fitur_kolom].values
y = data['Dominant_Style'].values

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)
jumlah_kelas = 4

# Bagi data: 80% train, 10% validasi, 10% uji
X_latih, X_sisa, y_latih, y_sisa = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
X_validasi, X_uji, y_validasi, y_uji = train_test_split(X_sisa, y_sisa, test_size=0.5, random_state=42, stratify=y_sisa)

# Normalisasi (standarisasi) hanya berdasarkan data latih
penskala = StandardScaler()
X_latih = penskala.fit_transform(X_latih)
X_validasi = penskala.transform(X_validasi)
X_uji = penskala.transform(X_uji)

print(f"Data latih: {X_latih.shape}, Data validasi: {X_validasi.shape}, Data uji: {X_uji.shape}")

Data latih: (904, 20), Data validasi: (113, 20), Data uji: (113, 20)


In [4]:
# Cell 4: Custom loss function untuk menangani ketidakseimbangan kelas
class FocalLoss(keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.25, name='focal_loss'):
        super().__init__(name=name)
        self.gamma = gamma
        self.alpha = alpha
    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_true_one_hot = tf.one_hot(y_true, depth=jumlah_kelas)
        ce = tf.keras.losses.categorical_crossentropy(y_true_one_hot, y_pred, from_logits=False)
        p_t = tf.exp(-ce)
        bobot_focal = self.alpha * (1 - p_t) ** self.gamma
        return tf.reduce_mean(bobot_focal * ce)
    def get_config(self):
        return {'gamma': self.gamma, 'alpha': self.alpha}

In [5]:
# Cell 5: Custom callback untuk mencetak loss/akurasi setiap 10 epoch, dan TensorBoard
class PencetakEpoch(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}: loss={logs['loss']:.4f}, akurasi={logs['accuracy']:.4f}, val_loss={logs['val_loss']:.4f}, val_akurasi={logs['val_accuracy']:.4f}")

# TensorBoard untuk memantau training
waktu_log = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_cb = keras.callbacks.TensorBoard(log_dir=waktu_log, histogram_freq=1)

In [6]:
# Cell 6: Arsitektur model dengan Functional API
def buat_model():
    masukan = Input(shape=(20,))
    x = layers.Dense(128, activation='relu')(masukan)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation='relu')(x)
    keluaran = layers.Dense(jumlah_kelas, activation='softmax')(x)
    model = Model(masukan, keluaran)
    model.compile(optimizer=keras.optimizers.Adam(0.001),
                  loss=FocalLoss(gamma=2.0, alpha=0.25),
                  metrics=['accuracy'])
    return model

In [7]:
# Cell 7: Latih satu model, simpan yang terbaik, kembalikan akurasi validasi
def latih_model(seed, epochs=120):
    tf.random.set_seed(seed)
    np.random.seed(seed)
    model = buat_model()
    pengecekan = keras.callbacks.ModelCheckpoint(f'model_seed_{seed}.keras',
                                                 monitor='val_accuracy',
                                                 save_best_only=True,
                                                 mode='max',
                                                 verbose=0)
    pengurang_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss',
                                                     factor=0.5,
                                                     patience=15,
                                                     min_lr=1e-7,
                                                     verbose=0)
    henti_awal = keras.callbacks.EarlyStopping(monitor='val_accuracy',
                                               patience=30,
                                               restore_best_weights=True,
                                               verbose=0)
    model.fit(X_latih, y_latih,
              validation_data=(X_validasi, y_validasi),
              epochs=epochs,
              batch_size=64,
              callbacks=[pengecekan, pengurang_lr, henti_awal, PencetakEpoch(), tensorboard_cb],
              verbose=0)
    model_terbaik = keras.models.load_model(f'model_seed_{seed}.keras',
                                            custom_objects={'FocalLoss': FocalLoss})
    pred_validasi = model_terbaik.predict(X_validasi, verbose=0)
    akurasi_validasi = accuracy_score(y_validasi, np.argmax(pred_validasi, axis=1))
    print(f"  Seed {seed:3d} -> akurasi_validasi: {akurasi_validasi:.4f}")
    return model_terbaik, akurasi_validasi

In [8]:
# Cell 8: Latih 3 model dengan seed berbeda, hitung bobot berdasarkan akurasi validasi
print("\nMelatih ensemble 3 model dengan seed 42, 123, 456...")
daftar_seed = [42, 123, 456]
kumpulan_model = []
kumpulan_akurasi_validasi = []
for seed in daftar_seed:
    print(f"  Training seed {seed}...")
    m, akurasi = latih_model(seed, epochs=120)
    kumpulan_model.append(m)
    kumpulan_akurasi_validasi.append(akurasi)
    print(f"    Selesai.")

bobot = np.array(kumpulan_akurasi_validasi) / np.sum(kumpulan_akurasi_validasi)
print(f"Bobot ensemble: {[f'{b:.3f}' for b in bobot]}")


Melatih ensemble 3 model dengan seed 42, 123, 456...
  Training seed 42...


I0000 00:00:1778677469.031460     202 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778677469.037712     202 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1778677473.137188     283 service.cc:152] XLA service 0x799b80013760 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778677473.137222     283 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1778677473.137226     283 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1778677473.620623     283 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1778677475.885781     283 device_compiler.h:188] Compiled clust

Epoch 10: loss=0.0611, akurasi=0.7876, val_loss=0.0717, val_akurasi=0.7788
Epoch 20: loss=0.0374, akurasi=0.8496, val_loss=0.0380, val_akurasi=0.8319
Epoch 30: loss=0.0267, akurasi=0.8971, val_loss=0.0336, val_akurasi=0.8673
Epoch 40: loss=0.0180, akurasi=0.9292, val_loss=0.0256, val_akurasi=0.8673
Epoch 50: loss=0.0139, akurasi=0.9358, val_loss=0.0299, val_akurasi=0.8850
Epoch 60: loss=0.0099, akurasi=0.9591, val_loss=0.0252, val_akurasi=0.8850
Epoch 70: loss=0.0121, akurasi=0.9491, val_loss=0.0265, val_akurasi=0.8938
Epoch 80: loss=0.0111, akurasi=0.9524, val_loss=0.0276, val_akurasi=0.9115
  Seed  42 -> akurasi_validasi: 0.9115
    Selesai.
  Training seed 123...
Epoch 10: loss=0.0597, akurasi=0.7777, val_loss=0.0746, val_akurasi=0.7168
Epoch 20: loss=0.0341, akurasi=0.8650, val_loss=0.0432, val_akurasi=0.8319
Epoch 30: loss=0.0193, akurasi=0.9270, val_loss=0.0337, val_akurasi=0.8584
Epoch 40: loss=0.0177, akurasi=0.9281, val_loss=0.0337, val_akurasi=0.8761
Epoch 50: loss=0.0117, ak

In [9]:
# Cell 9: Gabungkan prediksi dari ketiga model dengan bobot, hitung akurasi akhir
print("\nEvaluasi weighted ensemble pada data uji...")
probabilitas = np.zeros((len(X_uji), jumlah_kelas))
for m, w in zip(kumpulan_model, bobot):
    probabilitas += w * m.predict(X_uji, verbose=0)
y_pred = np.argmax(probabilitas, axis=1)
akurasi_uji = accuracy_score(y_uji, y_pred)
print(f"\nAkurasi ensemble pada data uji: {akurasi_uji:.4f} ({akurasi_uji*100:.2f}%)")

# Simpan model dengan akurasi validasi tertinggi sebagai model final
indeks_terbaik = np.argmax(kumpulan_akurasi_validasi)
model_final = kumpulan_model[indeks_terbaik]
model_final.save('learning_style_final.keras')
print(f"Model final disimpan sebagai 'learning_style_final.keras' (seed {daftar_seed[indeks_terbaik]})")


Evaluasi weighted ensemble pada data uji...

Akurasi ensemble pada data uji: 0.9381 (93.81%)
Model final disimpan sebagai 'learning_style_final.keras' (seed 42)


In [10]:
# Cell 10: Fungsi untuk memprediksi gaya belajar dari 20 nilai fitur
def prediksi_gaya_belajar(fitur):
    model_dimuat = keras.models.load_model('learning_style_final.keras',
                                           custom_objects={'FocalLoss': FocalLoss})
    fitur = np.array(fitur).reshape(1, -1)
    fitur_skala = penskala.transform(fitur)
    prob = model_dimuat.predict(fitur_skala, verbose=0)
    kelas = np.argmax(prob, axis=1)[0]
    return encoder.inverse_transform([kelas])[0]

# Contoh prediksi dengan sampel pertama dari data uji
contoh = X_uji[0]
label_asli = encoder.inverse_transform([y_uji[0]])[0]
label_hasil = prediksi_gaya_belajar(contoh)
print(f"Contoh inference: true = {label_asli}, predicted = {label_hasil}")

Contoh inference: true = Kinesthetic, predicted = Visual


In [11]:
# Cell 11: Tampilkan precision, recall, f1-score untuk setiap gaya belajar
print("\nLaporan Klasifikasi (Ensemble):")
print(classification_report(y_uji, y_pred, target_names=encoder.classes_))

print(f"\nTensorBoard log disimpan di: {waktu_log}")
print("Jalankan: tensorboard --logdir logs/fit (jika lokal)")


Laporan Klasifikasi (Ensemble):
              precision    recall  f1-score   support

    Auditory       0.96      0.83      0.89        29
 Kinesthetic       0.89      1.00      0.94        24
   ReadWrite       0.96      0.96      0.96        25
      Visual       0.94      0.97      0.96        35

    accuracy                           0.94       113
   macro avg       0.94      0.94      0.94       113
weighted avg       0.94      0.94      0.94       113


TensorBoard log disimpan di: logs/fit/20260513-130427
Jalankan: tensorboard --logdir logs/fit (jika lokal)
